# Data Exploration & SAS Validation — 09/15/2026

**What this notebook proves:** that we can rebuild one stage of the CMS Star Ratings pipeline in Python and get numbers that match SAS's output exactly — not just "close," but matching to essentially every decimal place.

## What's actually going on here

CMS's Star Ratings for hospitals are currently calculated by a program written in SAS (an older stats-focused programming language). We're rewriting that program in Python, but before doing the whole thing, we want to prove — on one small, low-stakes piece of it — that our Python code produces the exact same numbers SAS does.

That piece is called **measure standardization**, and it comes from the very first program in the original SAS pipeline: `0 - Data and Measure Standardization_2025Jul.sas`. In plain terms, that step does two things:

1. **Puts every measure on the same scale.** Hospitals are scored on all kinds of things — a death rate (like 12%) and a complication rate (like 180 per 1,000 surgeries) — that can't be compared directly because they're measured so differently. So for each measure, we convert every hospital's raw value into "how many notches above or below the average hospital is this?" (SAS's built-in tool for this is called `PROC STANDARD`.)
2. **Flips the sign where lower is actually better.** For measures like death rate, a *lower* number is the good outcome. So we flip those so that, afterward, "higher number" always means "better hospital" — consistently, across every measure.

## How we check our work

We compare our Python results against two files SAS already produced (our "answer key"):
- `MEASURE_AVERAGE_STDDEV_2025JUL.csv` — the average and standard deviation SAS calculated for each measure.
- `OUTCOME_MORTALITY.csv` — the final standardized values SAS calculated, per hospital.

This notebook also sets up the checklist we'll reuse for every later stage of the migration: (1) do we have the same rows/hospitals, (2) do the summary numbers (averages, etc.) match, (3) does every individual value match.

> Run this notebook from the repo's root folder. If the data folder can't be found, adjust the `DATA` path below.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# This just locates the "data/Project_1" folder automatically, whether this
# notebook is run from the repo's root folder or from inside notebooks/.
DATA = next((p / "data/Project_1" for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/Project_1").exists()), None)
assert DATA is not None, "Can't locate data/Project_1 — run this inside the repo."
print("data dir:", DATA)

## Step 1: Load the data

We need two things in memory before we can compare anything:
- **The input data** — the raw hospital numbers that go *into* SAS's calculation.
- **SAS's actual output** — the numbers that come *out* the other end, which we're treating as the "answer key" our Python code needs to match.

The cell below loads three CSV files:
1. `alldata_2025jul.csv` — the raw input data. One row per hospital, one column per measure.
2. `MEASURE_AVERAGE_STDDEV_2025JUL.csv` — SAS's calculated average and standard deviation for each measure (part of the answer key).
3. `OUTCOME_MORTALITY.csv` — SAS's final standardized mortality values, per hospital (the other part of the answer key — this is what we're ultimately trying to reproduce).


In [2]:
inp  = pd.read_csv(DATA / "Starrating/alldata_2025jul.csv")
stat = pd.read_csv(DATA / "SAS Output CSV/MEASURE_AVERAGE_STDDEV_2025JUL.csv")
mort_out = pd.read_csv(DATA / "SAS Output CSV/OUTCOME_MORTALITY.csv")

print("input alldata      :", inp.shape)
print("measure avg/stddev :", stat.shape)
print("outcome_mortality  :", mort_out.shape)
inp.head(3)


input alldata      : (4566, 93)
measure avg/stddev : (5, 49)
outcome_mortality  : (4566, 21)


,PROVIDER_ID,COMP_HIP_KNEE,IMM_3_DEN,EDAC_30_AMI,EDAC_30_AMI_DEN,READM_30_HOSP_WIDE_DEN,HAI_3,EDAC_30_HF_DEN,OP_18B_DEN,HAI_6,...,PSI_4_SURG_COMP,OP_35_ADM_DEN,SEP_1_DEN,EDAC_30_PN_DEN,HAI_5,HAI_4,HAI_4_DEN_VOL,HAI_4_DEN_PRED,H_INDI_STAR_RATING,H_GLOB_STAR_RATING
0,010001,0.030,4115.0,-13.8,296.0,2924.0,1.209,679.0,345.0,0.491,...,194.78,202.0,131.0,490.0,0.445,NaN,NaN,NaN,3.0,4.0
1,010005,0.027,2407.0,NaN,NaN,1056.0,0.000,176.0,1154.0,0.631,...,191.14,107.0,288.0,305.0,2.452,NaN,NaN,NaN,3.5,3.0
2,010006,0.040,2560.0,13.4,315.0,2560.0,0.000,508.0,349.0,0.026,...,201.13,NaN,162.0,621.0,0.371,0.0,116.0,1.086,2.5,2.0


## Step 2: Make sure we're comparing apples to apples (same hospitals)

Before comparing any numbers, we need to check that SAS and Python are working with the *same set of hospitals*. If SAS quietly threw out some hospitals before doing its calculation, our row counts wouldn't match up, and every comparison after that would be meaningless — we'd be comparing different sets of hospitals without realizing it.

**Why SAS might drop hospitals:** if a hospital doesn't report enough data for a given measure, SAS excludes that measure. And if, after excluding low-data measures, a hospital has *nothing* left to score it on, SAS drops that hospital entirely. That's just a rule baked into the original program — nothing we need to reproduce ourselves, just something we need to be aware of.

**For this specific batch of data (2025, July):** it turns out this didn't end up removing anything — no measures were dropped for having too little data, and no hospital ended up empty. So we *expect* the input row count and SAS's output row count to come out identical.

We check that below rather than just assuming it, because for a different batch of data, this assumption could turn out to be wrong — and we'd want this check to catch that automatically instead of silently giving bad results.


In [3]:
less100 = pd.read_csv(DATA / "SAS Output CSV/LESS100_MEASURE.csv")
print("measures excluded (volume<=100):", len(less100))          # expect 0 this quarter
print("input rows:", len(inp), " | SAS analysis-file rows:", len(mort_out))
assert len(inp) == len(mort_out), "Row counts differ — keep_hos DID drop hospitals this quarter!"
print("Row-count validation: PASS")


measures excluded (volume<=100): 0
input rows: 4566  | SAS analysis-file rows: 4566
Row-count validation: PASS


## Step 3: Recreate SAS's calculation in Python

We're focusing on the 7 **"Outcomes – Mortality"** measures — things like "30-day death rate after a heart attack" or "...after a stroke." For all 7 of these, **lower is better**: a hospital with a *lower* death rate is doing a *better* job. So after we calculate each measure's z-score, we flip its sign for all 7 of them (multiply by -1).

**The formula, spelled out in plain English**, for one measure, one hospital:
```
z-score = (this hospital's value − the average value across all hospitals) / (how spread out all hospitals' values are)
```
Then, because these are "lower is better" measures, we flip the sign:
```
final score = -1 × z-score
```
So a hospital with a very low (good) death rate ends up with a *positive* final score, and a hospital with a high (bad) death rate ends up with a *negative* score. Now, across every single measure in the whole pipeline — not just these 7 — "higher score" always means "better hospital." That consistency is the whole point of doing this.

**A small technical detail worth flagging, because it's an easy way to get subtly wrong answers:** "how spread out are the values" (the standard deviation) can technically be calculated two slightly different ways depending on a small denominator choice (dividing by the count of hospitals, `n`, vs. `n - 1`). SAS's built-in standardizing function (called `PROC STANDARD`, if you ever look at the original code) uses the `n - 1` version. Conveniently, that's also pandas' default behavior, so we don't have to change anything — but it's the kind of detail that silently breaks a migration if nobody checks it, so we're calling it out explicitly. Also: if a hospital is missing a value for some measure, we leave it as missing rather than guessing or filling it in — exactly what SAS does too.


In [ ]:
MORTALITY = ['MORT_30_AMI', 'MORT_30_CABG', 'MORT_30_COPD', 'MORT_30_HF',
             'MORT_30_PN', 'MORT_30_STK', 'PSI_4_SURG_COMP']

# All 7 mortality measures are "lower is better" -> every one gets sign-flipped.
FLIP = set(MORTALITY)

def standardize(series, flip):
    # Step A: z-score = (value - mean) / standard deviation.
    z = (series - series.mean()) / series.std(ddof=1)   # ddof=1 == SAS PROC STANDARD
    # Step B: flip the sign so "higher = better" for every measure.
    return -z if flip else z

py = inp[['PROVIDER_ID']].copy()
for m in MORTALITY:
    py[f'std_{m}'] = standardize(inp[m], flip=(m in FLIP))

py.head(3)


## Step 4: Check #1 — do our averages and "spread" numbers match SAS's?

Before checking every single hospital's score one by one, let's check something simpler and faster first: for each of the 7 measures, does the average value *we* calculated match the average value *SAS* calculated? Same question for the standard deviation (the "spread" number). If these don't match, there's no point checking anything more detailed — it'd mean something fundamental is already off.

The SAS output file (`MEASURE_AVERAGE_STDDEV_2025JUL.csv`) is organized a little unusually: instead of one column per statistic, it has a column called `_STAT_` that labels each row as `MEAN`, `STD`, etc. So to get "the average of measure X," you look for the row labeled `MEAN` and read the column for measure X. We pull out the `MEAN` and `STD` rows and line them up against what pandas calculates from the raw data.

We allow for a tiny difference (less than one-millionth) instead of requiring a perfect, digit-for-digit match — that's because numbers lose a sliver of precision when they get written out to a CSV file and read back in. That tiny bit of rounding is expected and not a real problem.


In [5]:
sm = stat.set_index('_STAT_')
rows = []
for m in MORTALITY:
    rows.append({
        'measure': m,
        'py_mean': inp[m].mean(),   'sas_mean': sm.loc['MEAN', m],
        'py_std':  inp[m].std(ddof=1), 'sas_std': sm.loc['STD', m],
    })
chk = pd.DataFrame(rows)
chk['mean_diff'] = (chk.py_mean - chk.sas_mean).abs()
chk['std_diff']  = (chk.py_std  - chk.sas_std ).abs()
print("max mean diff:", chk.mean_diff.max())
print("max std  diff:", chk.std_diff.max())
assert chk.mean_diff.max() < 1e-6 and chk.std_diff.max() < 1e-6, "Mean/std mismatch!"
print("Aggregate-statistic validation: PASS")
chk.round(6)


max mean diff: 1.1408474165364169e-09
max std  diff: 2.8196112111800176e-10
Aggregate-statistic validation: PASS


,measure,py_mean,sas_mean,py_std,sas_std,mean_diff,std_diff
0,MORT_30_AMI,0.125946,0.125946,0.012764,0.012764,0.0,0.0
1,MORT_30_CABG,0.029315,0.029315,0.008308,0.008308,0.0,0.0
2,MORT_30_COPD,0.094269,0.094269,0.015095,0.015095,0.0,0.0
3,MORT_30_HF,0.119070,0.119070,0.021616,0.021616,0.0,0.0
4,MORT_30_PN,0.179871,0.179871,0.028947,0.028947,0.0,0.0
5,MORT_30_STK,0.137445,0.137445,0.018728,0.018728,0.0,0.0
6,PSI_4_SURG_COMP,177.704353,177.704353,23.324928,23.324928,0.0,0.0


## Step 5: Check #2 — does every individual hospital's score match?

Matching averages is a good sign, but it's not real proof — two very different sets of numbers can still average out to the same thing. The real test is: for *every* hospital, and *every* one of the 7 measures, does our calculated score match SAS's score, down to the decimal?

We do this by lining up our results next to SAS's results, matched on hospital ID (`PROVIDER_ID`), then comparing each pair of numbers one by one. We're checking two separate things:
1. **How close are the two numbers?** (We expect a tiny rounding difference — the same kind of CSV rounding noise as before — not a real disagreement.)
2. **Do we agree on which values are missing?** If SAS has no value for a given hospital/measure (because that hospital didn't report enough data), we should have no value there either — not a value we made up, and not a gap where SAS actually had a number.


In [6]:
sas = mort_out[['PROVIDER_ID'] + [f'std_{m}' for m in MORTALITY]]
merged = py.merge(sas, on='PROVIDER_ID', suffixes=('_py', '_sas'))
assert len(merged) == len(py) == len(sas), "Join dropped rows — PROVIDER_ID mismatch!"

report = []
for m in MORTALITY:
    a, b = merged[f'std_{m}_py'], merged[f'std_{m}_sas']
    diff = (a - b).abs()
    report.append({
        'measure': m,
        'max_abs_diff': diff.max(),
        'missing_py':  int(a.isna().sum()),
        'missing_sas': int(b.isna().sum()),
        'missing_match': bool((a.isna() == b.isna()).all()),
    })
report = pd.DataFrame(report)
worst = report.max_abs_diff.max()
print(f"Worst absolute difference across all cells: {worst:.2e}")
assert worst < 1e-6, "Values diverge beyond float-rounding tolerance!"
assert report.missing_match.all(), "Missing-value patterns differ!"
print("Column-level value validation: PASS  (differences are just CSV float rounding)")
report


Worst absolute difference across all cells: 5.00e-10
Column-level value validation: PASS  (differences are just CSV float rounding)


,measure,max_abs_diff,missing_py,missing_sas,missing_match
0,MORT_30_AMI,4.693463e-10,2564,2564,True
1,MORT_30_CABG,4.999481e-10,3677,3677,True
2,MORT_30_COPD,4.998841e-10,1919,1919,True
3,MORT_30_HF,4.975000e-10,1423,1423,True
4,MORT_30_PN,4.969007e-10,937,937,True
5,MORT_30_STK,4.838827e-10,2353,2353,True
6,PSI_4_SURG_COMP,4.988719e-10,2997,2997,True


## Step 6: What we found

All three checks passed for the 7 mortality measures:

| Check | What it confirms | Result |
|---|---|---|
| Row count | SAS and Python are looking at the same 4,566 hospitals | ✅ Pass |
| Averages & standard deviations | Our summary math matches SAS's, measure by measure | ✅ Pass (matches to within 0.000001) |
| Individual hospital scores | Every single standardized value matches SAS's, hospital by hospital | ✅ Pass (matches to within 0.0000000005 — essentially zero) |
| Missing-value pattern | We're not missing data SAS has, and we don't have data SAS is missing | ✅ Pass |

### What this actually means

- We've now proven, with real data, that a simple formula — `(value − average) / spread`, with the sign flipped for "lower is better" measures — is a faithful Python translation of what SAS's built-in standardizing function does. Not "close enough," but matching to a dozen decimal places.
- The tiny differences we did see (way less than a millionth) just come from numbers losing a sliver of precision when SAS writes them to a CSV file — not from any actual mistake in the math. So going forward, a tolerance of about 0.000001 is a reasonable "close enough to count as matching" threshold for this whole project.
- More importantly, we now have a reusable, 3-step recipe for proving *any* future piece of this migration is correct: **(1) check we have the same rows, (2) check the summary numbers match, (3) check every individual value matches.** The next notebooks will reuse this exact pattern instead of reinventing it.

### What's next

- **Cover the rest of the measures:** we only tested the 7 mortality measures here. The full pipeline standardizes around 48 measures total, and we'll need to repeat this same check for the others (the original SAS file lists, for every measure, whether "lower is better," so we know which ones to sign-flip).
- **Move on to file #1 in the pipeline:** once every individual measure is standardized, the next SAS program averages related measures together into group scores — e.g., all the safety-related measures become one "Safety" score. We'll build a notebook that validates that step next, checking against files like `OUTCOME_SAFETY.csv`, `OUTCOME_READMISSION.csv`, etc.
- **Save the trickiest part for last:** the very last step in the pipeline sorts hospitals into their final 1–5 star rating using a clustering algorithm (a way of grouping similar hospitals together, done in SAS via something called `PROC FASTCLUS`). Reproducing that step exactly is the hardest part of this whole migration, so it deserves its own dedicated notebook rather than being folded into this one.
